# Phase 5 — Explainable AI

**Project:** AI Mental Health Assessment Platform  
**Goal:** Explain the trained models using Feature Importance, SHAP, and Partial Dependence Plots.

This notebook explains three saved models:

1. Mental Health Condition Prediction
2. Severity Prediction
3. Treatment Recommendation

> Important: This project is for educational and portfolio purposes only. It is not a medical diagnosis system.

## Step 1 — Import Libraries

We import ML evaluation, plotting, model loading, and explainability libraries.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib

from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, f1_score

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as e:
    SHAP_AVAILABLE = False
    print("SHAP is not installed or failed to import.")
    print("Install with: pip install shap")
    print("Details:", e)

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## Step 2 — Define Project Paths

This block keeps the notebook compatible with your local repo structure.

In [ ]:
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
EXPLAIN_DIR = REPORTS_DIR / "explainability"

for directory in [REPORTS_DIR, FIGURES_DIR, EXPLAIN_DIR, FIGURES_DIR / "feature_importance", FIGURES_DIR / "shap", FIGURES_DIR / "pdp"]:
    directory.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

## Step 3 — Define Target Configuration

Each key represents one prediction task.

In [ ]:
TARGET_CONFIG = {
    "condition": {
        "display_name": "Mental Health Condition",
        "model_file": "condition_best_model.pkl",
        "data_folder": "condition"
    },
    "severity": {
        "display_name": "Severity Level",
        "model_file": "severity_best_model.pkl",
        "data_folder": "severity"
    },
    "treatment": {
        "display_name": "Recommended Treatment",
        "model_file": "treatment_best_model.pkl",
        "data_folder": "treatment"
    }
}

## Step 4 — Helper Function: Load Processed Data

This function loads `X_train`, `X_test`, `y_train`, and `y_test` from Phase 2 outputs.

In [ ]:
def load_processed_target_data(target_key):
    target_dir = PROCESSED_DIR / TARGET_CONFIG[target_key]["data_folder"]
    files = {
        "X_train": target_dir / "X_train.csv",
        "X_test": target_dir / "X_test.csv",
        "y_train": target_dir / "y_train.csv",
        "y_test": target_dir / "y_test.csv",
    }
    missing_files = [str(path) for path in files.values() if not path.exists()]
    if missing_files:
        raise FileNotFoundError(
            "Phase 2 processed data files are missing. Run notebooks/02_data_preprocessing.ipynb first.

"
            "Missing files:
" + "
".join(missing_files)
        )
    X_train = pd.read_csv(files["X_train"])
    X_test = pd.read_csv(files["X_test"])
    y_train = pd.read_csv(files["y_train"]).squeeze("columns")
    y_test = pd.read_csv(files["y_test"]).squeeze("columns")
    return X_train, X_test, y_train, y_test

## Step 5 — Helper Function: Load Best Model

This loads the saved best model from Phase 3.

In [ ]:
def load_best_model(target_key):
    model_path = MODELS_DIR / TARGET_CONFIG[target_key]["model_file"]
    if not model_path.exists():
        raise FileNotFoundError(
            f"Best model not found: {model_path}
"
            "Please run notebooks/03_model_training.ipynb first."
        )
    return joblib.load(model_path)

## Step 6 — Load All Models and Data

We load all three models and their test datasets.

In [ ]:
project_data = {}

for target_key in TARGET_CONFIG:
    print(f"Loading: {target_key}")
    X_train, X_test, y_train, y_test = load_processed_target_data(target_key)
    model = load_best_model(target_key)
    project_data[target_key] = {
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }
    print("  X_train:", X_train.shape, "X_test:", X_test.shape)
    print("  Model:", type(model).__name__)

## Step 7 — Baseline Prediction Check

Before explanation, confirm the models predict properly.

In [ ]:
quick_scores = []

for target_key, item in project_data.items():
    model = item["model"]
    X_test = item["X_test"]
    y_test = item["y_test"]
    y_pred = model.predict(X_test)
    quick_scores.append({
        "target": target_key,
        "task": TARGET_CONFIG[target_key]["display_name"],
        "accuracy": accuracy_score(y_test, y_pred),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    })

quick_scores_df = pd.DataFrame(quick_scores)
quick_scores_df

## Step 8 — Built-in Feature Importance Function

Tree-based models usually provide `.feature_importances_`. Linear models may provide `.coef_`.

In [ ]:
def get_builtin_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    elif hasattr(model, "coef_"):
        coef = model.coef_
        importance = np.mean(np.abs(coef), axis=0) if coef.ndim > 1 else np.abs(coef)
    else:
        return None

    return (
        pd.DataFrame({"feature": feature_names, "importance": importance})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

## Step 9 — Plot Feature Importance Function

In [ ]:
def plot_top_feature_importance(importance_df, target_key, top_n=20):
    if importance_df is None or importance_df.empty:
        print(f"No built-in feature importance available for {target_key}")
        return None

    top_df = importance_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_df["feature"], top_df["importance"])
    plt.title(f"Top {top_n} Feature Importance — {TARGET_CONFIG[target_key]['display_name']}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()

    output_path = FIGURES_DIR / "feature_importance" / f"{target_key}_feature_importance.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    return output_path

## Step 10 — Generate Built-in Feature Importance

In [ ]:
builtin_importance_results = {}

for target_key, item in project_data.items():
    model = item["model"]
    X_train = item["X_train"]
    importance_df = get_builtin_feature_importance(model, X_train.columns.tolist())
    builtin_importance_results[target_key] = importance_df

    if importance_df is not None:
        output_csv = EXPLAIN_DIR / f"{target_key}_builtin_feature_importance.csv"
        importance_df.to_csv(output_csv, index=False)
        print(f"Saved: {output_csv}")
        display(importance_df.head(15))
        plot_top_feature_importance(importance_df, target_key, top_n=20)
    else:
        print(f"No built-in importance for {target_key}: {type(model).__name__}")

## Step 11 — Permutation Importance

Permutation importance works for many model types. It measures how much model score drops after randomly shuffling one feature.

In [ ]:
def calculate_permutation_importance(model, X_test, y_test, feature_names, target_key, n_repeats=10):
    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
        scoring="f1_weighted",
        n_jobs=-1
    )
    perm_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)

    output_csv = EXPLAIN_DIR / f"{target_key}_permutation_importance.csv"
    perm_df.to_csv(output_csv, index=False)
    return perm_df

## Step 12 — Generate Permutation Importance

In [ ]:
permutation_results = {}

for target_key, item in project_data.items():
    print(f"Permutation importance for: {target_key}")
    perm_df = calculate_permutation_importance(
        model=item["model"],
        X_test=item["X_test"],
        y_test=item["y_test"],
        feature_names=item["X_test"].columns.tolist(),
        target_key=target_key,
        n_repeats=10
    )
    permutation_results[target_key] = perm_df
    display(perm_df.head(15))

## Step 13 — Plot Permutation Importance

In [ ]:
def plot_permutation_importance(perm_df, target_key, top_n=20):
    top_df = perm_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top_df["feature"], top_df["importance_mean"], xerr=top_df["importance_std"])
    plt.title(f"Permutation Importance — {TARGET_CONFIG[target_key]['display_name']}")
    plt.xlabel("Mean F1 Score Drop")
    plt.ylabel("Feature")
    plt.tight_layout()

    output_path = FIGURES_DIR / "feature_importance" / f"{target_key}_permutation_importance.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    return output_path

for target_key, perm_df in permutation_results.items():
    plot_permutation_importance(perm_df, target_key, top_n=20)

## Step 14 — SHAP Helper Functions

SHAP can be slow. This notebook uses a sample from `X_test` to keep runtime manageable.

In [ ]:
def sample_for_explainability(X, max_rows=100):
    if len(X) <= max_rows:
        return X.copy()
    return X.sample(max_rows, random_state=RANDOM_STATE)


def compute_shap_values(model, X_sample):
    # TreeExplainer is fast for tree-based models.
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        return explainer, shap_values
    except Exception:
        # Fallback for other models. Slower but more general.
        background = shap.sample(X_sample, min(50, len(X_sample)), random_state=RANDOM_STATE)
        explainer = shap.Explainer(model.predict, background)
        shap_values = explainer(X_sample)
        return explainer, shap_values

## Step 15 — Generate SHAP Summary Plots

For multiclass models, SHAP values can have different structures depending on model type. This function handles common cases.

In [ ]:
def plot_shap_summary(model, X_sample, target_key):
    if not SHAP_AVAILABLE:
        print("SHAP is not available. Skipping SHAP summary plot.")
        return None

    print(f"Computing SHAP for {target_key} with sample size {len(X_sample)}...")
    explainer, shap_values = compute_shap_values(model, X_sample)

    output_path = FIGURES_DIR / "shap" / f"{target_key}_shap_summary.png"

    plt.figure()
    try:
        # Common multiclass tree model: list of arrays
        if isinstance(shap_values, list):
            values_to_plot = np.mean([np.abs(v) for v in shap_values], axis=0)
            shap.summary_plot(values_to_plot, X_sample, show=False, plot_type="bar")
        else:
            # New SHAP Explanation object or array
            shap.summary_plot(shap_values, X_sample, show=False, plot_type="bar")

        plt.title(f"SHAP Summary — {TARGET_CONFIG[target_key]['display_name']}")
        plt.tight_layout()
        plt.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.show()
        return output_path
    except Exception as e:
        print(f"Could not create SHAP summary for {target_key}:", e)
        plt.close()
        return None

## Step 16 — Run SHAP Summary for All Targets

In [ ]:
shap_summary_paths = {}

for target_key, item in project_data.items():
    X_sample = sample_for_explainability(item["X_test"], max_rows=100)
    shap_summary_paths[target_key] = plot_shap_summary(item["model"], X_sample, target_key)

## Step 17 — SHAP Local Explanation for One Prediction

This step tries to explain one individual prediction. If the model/SHAP version does not support it, the notebook will skip safely.

In [ ]:
def explain_single_prediction_with_shap(model, X_sample, target_key, row_index=0):
    if not SHAP_AVAILABLE:
        print("SHAP is not available. Skipping local explanation.")
        return None

    try:
        explainer, shap_values = compute_shap_values(model, X_sample)
        single_row = X_sample.iloc[[row_index]]
        pred = model.predict(single_row)[0]
        print(f"Target: {target_key}")
        print(f"Prediction for row {row_index}: {pred}")

        # Save force plot as HTML when possible.
        output_html = EXPLAIN_DIR / f"{target_key}_single_prediction_shap.html"
        shap.initjs()

        if isinstance(shap_values, list):
            class_labels = list(model.classes_) if hasattr(model, "classes_") else list(range(len(shap_values)))
            pred_class_index = class_labels.index(pred) if pred in class_labels else 0
            force_plot = shap.force_plot(
                explainer.expected_value[pred_class_index],
                shap_values[pred_class_index][row_index, :],
                single_row,
                matplotlib=False
            )
        else:
            sv = explainer(single_row) if callable(explainer) else shap_values[row_index]
            force_plot = shap.plots.force(sv, matplotlib=False)

        shap.save_html(str(output_html), force_plot)
        print("Saved:", output_html)
        return output_html
    except Exception as e:
        print(f"Could not create local SHAP explanation for {target_key}:", e)
        return None

single_shap_outputs = {}
for target_key, item in project_data.items():
    X_sample = sample_for_explainability(item["X_test"], max_rows=50)
    single_shap_outputs[target_key] = explain_single_prediction_with_shap(item["model"], X_sample, target_key, row_index=0)

## Step 18 — Select Top Features for PDP

Partial Dependence Plots show how prediction changes when one feature changes.

In [ ]:
def get_top_features_for_pdp(target_key, top_n=4):
    if target_key in permutation_results and not permutation_results[target_key].empty:
        return permutation_results[target_key]["feature"].head(top_n).tolist()
    if builtin_importance_results.get(target_key) is not None:
        return builtin_importance_results[target_key]["feature"].head(top_n).tolist()
    return project_data[target_key]["X_test"].columns[:top_n].tolist()

for target_key in TARGET_CONFIG:
    print(target_key, "→", get_top_features_for_pdp(target_key, top_n=4))

## Step 19 — Generate Partial Dependence Plots

PDP works best with numerical features. Because Phase 2 already encoded/scaled data, the plots explain processed feature behavior.

In [ ]:
def plot_partial_dependence(model, X, target_key, features):
    saved_paths = []
    for feature in features:
        try:
            fig, ax = plt.subplots(figsize=(7, 5))
            PartialDependenceDisplay.from_estimator(
                model,
                X,
                features=[feature],
                ax=ax,
                grid_resolution=20
            )
            ax.set_title(f"PDP: {feature} — {TARGET_CONFIG[target_key]['display_name']}")
            plt.tight_layout()
            output_path = FIGURES_DIR / "pdp" / f"{target_key}_pdp_{feature}.png"
            plt.savefig(output_path, dpi=150, bbox_inches="tight")
            plt.show()
            saved_paths.append(output_path)
        except Exception as e:
            print(f"Could not create PDP for {target_key} - {feature}:", e)
    return saved_paths

pdp_paths = {}
for target_key, item in project_data.items():
    top_features = get_top_features_for_pdp(target_key, top_n=4)
    X_sample = sample_for_explainability(item["X_test"], max_rows=200)
    pdp_paths[target_key] = plot_partial_dependence(item["model"], X_sample, target_key, top_features)

## Step 20 — Compare Top Features Across Targets

In [ ]:
top_feature_rows = []

for target_key in TARGET_CONFIG:
    source = "permutation_importance"
    df_imp = permutation_results.get(target_key)
    if df_imp is None or df_imp.empty:
        source = "builtin_importance"
        df_imp = builtin_importance_results.get(target_key)
        if df_imp is None:
            continue
        feature_col = "feature"
        importance_col = "importance"
    else:
        feature_col = "feature"
        importance_col = "importance_mean"

    for rank, row in df_imp.head(10).iterrows():
        top_feature_rows.append({
            "target": target_key,
            "task": TARGET_CONFIG[target_key]["display_name"],
            "rank": rank + 1,
            "feature": row[feature_col],
            "importance": row[importance_col],
            "source": source,
        })

top_features_df = pd.DataFrame(top_feature_rows)
top_features_output = EXPLAIN_DIR / "top_features_across_targets.csv"
top_features_df.to_csv(top_features_output, index=False)
top_features_df.head(30)

## Step 21 — Save Explainability Metadata

In [ ]:
metadata = {
    "phase": "Phase 5 - Explainable AI",
    "targets": list(TARGET_CONFIG.keys()),
    "methods": [
        "Built-in feature importance",
        "Permutation importance",
        "SHAP summary plot",
        "SHAP local explanation",
        "Partial dependence plots"
    ],
    "outputs": {
        "explainability_dir": str(EXPLAIN_DIR),
        "figures_dir": str(FIGURES_DIR),
    },
    "note": "This is an educational explainability workflow, not a medical diagnostic explanation."
}

metadata_path = EXPLAIN_DIR / "explainability_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

metadata_path

## Step 22 — Generate Explainability Report

In [ ]:
report_path = REPORTS_DIR / "explainability_report.md"

summary_lines = []
for target_key in TARGET_CONFIG:
    summary_lines.append(f"## {TARGET_CONFIG[target_key]['display_name']}")
    df_imp = permutation_results.get(target_key)
    if df_imp is not None and not df_imp.empty:
        top_features = df_imp.head(5)["feature"].tolist()
        summary_lines.append("Top features by permutation importance:")
        for i, feature in enumerate(top_features, start=1):
            summary_lines.append(f"{i}. {feature}")
    summary_lines.append("")

report = f"""# Phase 5 — Explainable AI Report

## Project
AI Mental Health Assessment Platform

## Purpose
This report summarizes explainability outputs for the three prediction tasks:

1. Mental Health Condition Prediction
2. Severity Prediction
3. Treatment Recommendation

## Methods Used
- Built-in feature importance
- Permutation importance
- SHAP summary plot
- SHAP local explanation
- Partial Dependence Plot

## Important Disclaimer
This model is for educational and portfolio demonstration only. It should not be used as a medical diagnosis tool.

{chr(10).join(summary_lines)}

## Saved Outputs
- `reports/explainability/`
- `reports/figures/feature_importance/`
- `reports/figures/shap/`
- `reports/figures/pdp/`

## Next Phase
Phase 6 will build a reusable prediction pipeline for FastAPI and Streamlit deployment.
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("Saved:", report_path)

## Step 23 — Final Checklist

After running this notebook, check these outputs:

```text
reports/
├── explainability_report.md
├── explainability/
│   ├── *_builtin_feature_importance.csv
│   ├── *_permutation_importance.csv
│   ├── *_single_prediction_shap.html
│   ├── top_features_across_targets.csv
│   └── explainability_metadata.json
└── figures/
    ├── feature_importance/
    ├── shap/
    └── pdp/
```

In [ ]:
print("Phase 5 completed successfully.")
print("Report:", report_path)
print("Explainability outputs:", EXPLAIN_DIR)
print("Figures:", FIGURES_DIR)

# Phase 5 Conclusion

In this phase, we explained the three trained models using feature importance, permutation importance, SHAP, and PDP. These outputs help us understand which lifestyle and mental health indicators influence model predictions most.

Next: **Phase 6 — Prediction Pipeline**.